In [2]:

from sqlalchemy import create_engine

# Replace 'your_mac_username' with your actual Mac login name
engine = create_engine("postgresql://scoopcalendar@localhost:5432/nba")

# Test connection
conn = engine.connect()
print("Connected successfully!")
conn.close()

Connected successfully!


In [3]:
!pip install nba_api pandas sqlalchemy psycopg2-binary tqdm

In [4]:
import time
import pandas as pd
from tqdm.notebook import tqdm
from sqlalchemy import create_engine, text
from nba_api.stats.endpoints import ShotChartDetail
from nba_api.stats.static import teams

In [5]:
DB_USER = "scoopcalendar"
DB_NAME = "nba"
DB_HOST = "localhost"
DB_PORT = "5432"

engine = create_engine(f"postgresql://{DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# Test connection
try:
    conn = engine.connect()
    print("✅ Connected to PostgreSQL successfully!")
    conn.close()
except Exception as e:
    print("❌ Connection failed:", e)

✅ Connected to PostgreSQL successfully!


In [6]:
create_table_sql = """
CREATE TABLE IF NOT EXISTS playoff_shots (
    id SERIAL PRIMARY KEY,
    game_id VARCHAR(20),
    game_event_id INTEGER,
    player_id INTEGER,
    player_name VARCHAR(100),
    team_id INTEGER,
    team_name VARCHAR(100),
    season VARCHAR(10),
    period INTEGER,
    minutes_remaining INTEGER,
    seconds_remaining INTEGER,
    loc_x INTEGER,
    loc_y INTEGER,
    shot_distance INTEGER,
    shot_attempted_flag INTEGER,
    shot_made_flag INTEGER,
    shot_type VARCHAR(50),
    action_type VARCHAR(100)
);
"""

with engine.connect() as conn:
    conn.execute(text(create_table_sql))
    print("✅ Table 'playoff_shots' is ready.")

✅ Table 'playoff_shots' is ready.


In [7]:
def fetch_shots(team_id, season_str):
    """
    Fetch exact shot locations using ShotChartDetail with the correct season_nullable keyword.
    Skips missing/unavailable data safely.
    """
    try:
        shot_data = ShotChartDetail(
            team_id=team_id,
            season_nullable=season_str,    # ✅ Correct keyword
            season_type_all_star="Playoffs",
            league_id="00",
            context_measure_simple="FGA",
            player_id=0                     # All players
        )
        df = shot_data.get_data_frames()[0]
        return df
    except Exception as e:
        print(f"⚠ Skipping Team {team_id}, Season {season_str}: {e}")
        return None

In [8]:
team_list = teams.get_teams()
START_YEAR = 2013
END_YEAR = 2025  # adjust to latest season

for year in range(START_YEAR, END_YEAR + 1):
    season_str = f"{year}-{str(year+1)[2:]}"
    print(f"\n📅 Downloading {season_str} Playoffs...")

    for team in tqdm(team_list, desc=f"{season_str} teams"):
        team_id = team["id"]
        team_name = team["full_name"]

        df = fetch_shots(team_id, season_str)
        if df is not None and not df.empty:
            df["TEAM_NAME"] = team_name
            df["SEASON"] = season_str

            df = df[[
                "GAME_ID",
                "GAME_EVENT_ID",
                "PLAYER_ID",
                "PLAYER_NAME",
                "TEAM_ID",
                "TEAM_NAME",
                "SEASON",
                "PERIOD",
                "MINUTES_REMAINING",
                "SECONDS_REMAINING",
                "LOC_X",
                "LOC_Y",
                "SHOT_DISTANCE",
                "SHOT_ATTEMPTED_FLAG",
                "SHOT_MADE_FLAG",
                "SHOT_TYPE",
                "ACTION_TYPE"
            ]]

            df.columns = [
                "game_id",
                "game_event_id",
                "player_id",
                "player_name",
                "team_id",
                "team_name",
                "season",
                "period",
                "minutes_remaining",
                "seconds_remaining",
                "loc_x",
                "loc_y",
                "shot_distance",
                "shot_attempted_flag",
                "shot_made_flag",
                "shot_type",
                "action_type"
            ]

            try:
                df.to_sql(
                    "playoff_shots",
                    engine,
                    if_exists="append",
                    index=False,
                    method="multi",
                    chunksize=1000
                )
            except Exception as e:
                print(f"⚠ Database insert failed: {e}")

        time.sleep(0.6)  # prevent rate limiting


📅 Downloading 2013-14 Playoffs...


2013-14 teams:   0%|          | 0/30 [00:00<?, ?it/s]


📅 Downloading 2014-15 Playoffs...


2014-15 teams:   0%|          | 0/30 [00:00<?, ?it/s]


📅 Downloading 2015-16 Playoffs...


...

2023-24 teams:   0%|          | 0/30 [00:00<?, ?it/s]


📅 Downloading 2024-25 Playoffs...


2024-25 teams:   0%|          | 0/30 [00:00<?, ?it/s]


📅 Downloading 2025-26 Playoffs...


2025-26 teams:   0%|          | 0/30 [00:00<?, ?it/s]

In [9]:
index_sql = [
    "CREATE INDEX IF NOT EXISTS idx_game_id ON playoff_shots(game_id);",
    "CREATE INDEX IF NOT EXISTS idx_player_id ON playoff_shots(player_id);",
    "CREATE INDEX IF NOT EXISTS idx_season ON playoff_shots(season);",
    "CREATE INDEX IF NOT EXISTS idx_team_id ON playoff_shots(team_id);"
]

with engine.connect() as conn:
    for stmt in index_sql:
        conn.execute(text(stmt))

print("✅ Indexes created for faster queries.")

✅ Indexes created for faster queries.


In [10]:
query = """
SELECT player_name,
       COUNT(*) as attempts,
       SUM(shot_made_flag) as makes,
       ROUND(SUM(shot_made_flag)::numeric / COUNT(*), 3) as fg_pct
FROM playoff_shots
GROUP BY player_name
ORDER BY attempts DESC
LIMIT 10;
"""

pd.read_sql(query, engine)

,player_name,attempts,makes,fg_pct
0,LeBron James,6346,3288.0,0.518
1,Stephen Curry,5642,2572.0,0.456
2,Kevin Durant,4862,2352.0,0.484
3,Klay Thompson,4822,2100.0,0.436
4,James Harden,4522,1914.0,0.423
5,Jayson Tatum,4482,1966.0,0.439
6,Jaylen Brown,4060,1956.0,0.482
7,Jimmy Butler III,3794,1746.0,0.460
8,Kawhi Leonard,3768,1920.0,0.510
9,Nikola Jokić,3728,1956.0,0.525


In [22]:
query = """
SELECT game_id,
       season,
       team_name,
       loc_x,
       loc_y,
       shot_made_flag,
       shot_type,
       action_type
FROM playoff_shots
WHERE player_name = 'Kyle Korver'
ORDER BY season, game_id;
"""

pd.read_sql(query, engine)

,game_id,season,team_name,loc_x,loc_y,shot_made_flag,shot_type,action_type
0,0041300101,2013-14,Atlanta Hawks,17.0,254.0,0,3PT Field Goal,Jump Shot
1,0041300101,2013-14,Atlanta Hawks,-24.0,258.0,1,3PT Field Goal,Jump Shot
2,0041300101,2013-14,Atlanta Hawks,-229.0,3.0,0,3PT Field Goal,Jump Shot
3,0041300101,2013-14,Atlanta Hawks,105.0,175.0,0,2PT Field Goal,Jump Shot
4,0041300101,2013-14,Atlanta Hawks,-138.0,224.0,0,3PT Field Goal,Jump Shot
...,...,...,...,...,...,...,...,...
545,0041900204,2019-20,Milwaukee Bucks,188.0,157.0,1,3PT Field Goal,Jump Shot
546,0041900204,2019-20,Milwaukee Bucks,-230.0,-9.0,0,3PT Field Goal,Jump Shot
547,0041900204,2019-20,Milwaukee Bucks,154.0,154.0,0,2PT Field Goal,Jump Shot
548,0041900205,2019-20,Milwaukee Bucks,-202.0,180.0,1,3PT Field Goal,Pullup Jump shot


In [12]:
query = """
SELECT * FROM playoff_shots
"""

pd.read_sql(query, engine)

,game_id,game_event_id,player_id,player_name,team_id,team_name,season,period,minutes_remaining,seconds_remaining,loc_x,loc_y,shot_distance,shot_attempted_flag,shot_made_flag,shot_type,action_type
0,0041300101,3,2594,Kyle Korver,1610612737,Atlanta Hawks,2013-14,1,11,28,17.0,254.0,25.0,1,0,3PT Field Goal,Jump Shot
1,0041300101,7,200794,Paul Millsap,1610612737,Atlanta Hawks,2013-14,1,10,53,72.0,37.0,8.0,1,0,2PT Field Goal,Jump Shot
2,0041300101,9,203544,Pero Antic,1610612737,Atlanta Hawks,2013-14,1,10,51,-10.0,-17.0,1.0,1,1,2PT Field Goal,Layup Shot
3,0041300101,20,2594,Kyle Korver,1610612737,Atlanta Hawks,2013-14,1,9,50,-24.0,258.0,25.0,1,1,3PT Field Goal,Jump Shot
4,0041300101,23,201952,Jeff Teague,1610612737,Atlanta Hawks,2013-14,1,9,27,121.0,145.0,18.0,1,1,2PT Field Goal,Pullup Jump shot
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
339675,0042400126,614,1630595,Cade Cunningham,1610612765,Detroit Pistons,2024-25,4,3,26,71.0,21.0,7.0,1,0,2PT Field Goal,Driving Floating Jump Shot
339676,0042400126,623,203501,Tim Hardaway Jr.,1610612765,Detroit Pistons,2024-25,4,2,35,-167.0,1.0,16.0,1,1,2PT Field Goal,Pullup Jump shot
339677,0042400126,632,1631105,Jalen Duren,1610612765,Detroit Pistons,2024-25,4,1,57,64.0,56.0,8.0,1,0,2PT Field Goal,Alley Oop Layup shot
339678,0042400126,635,1630595,Cade Cunningham,1610612765,Detroit Pistons,2024-25,4,1,24,-32.0,1.0,3.0,1,0,2PT Field Goal,Driving Layup Shot
